In [21]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq

from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver # ram memory storage

In [22]:
load_dotenv()

llm = ChatGroq(model="moonshotai/kimi-k2-instruct-0905")


In [23]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [24]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [25]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [26]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver() # defined checkpoints

workflow = graph.compile(checkpointer=checkpointer)

In [27]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1) # given thread 

{'topic': 'pizza',
 'joke': 'Why did the pizza apply for a job?\n\nBecause it wanted to make some *dough* and finally get a *slice* of the good life!',
 'explanation': 'The joke is built on two layers of pizza-flavored wordplay:\n\n1. “Dough”  \n   In pizza terms, dough is literally what the crust is made of. In slang, “dough” has meant “money” since at least the 19th century. So the pizza is applying for a job to earn the thing it’s already made of—an identity gag.\n\n2. “A slice of the good life”  \n   Pizzas are served in slices, so the phrase “get a slice of the good life” is both the idiom for enjoying prosperity and a literal description of what every pizza already experiences—being sliced. The humor comes from the pizza wanting the metaphorical version of something it physically can’t escape.\n\nPut together, the punch line pretends the pizza has the same career anxieties as people: needing cash (dough) and dreaming of comfort (a slice of the good life), while every word in the 

In [28]:
workflow.get_state(config1) # to see is stored 

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it wanted to make some *dough* and finally get a *slice* of the good life!', 'explanation': 'The joke is built on two layers of pizza-flavored wordplay:\n\n1. “Dough”  \n   In pizza terms, dough is literally what the crust is made of. In slang, “dough” has meant “money” since at least the 19th century. So the pizza is applying for a job to earn the thing it’s already made of—an identity gag.\n\n2. “A slice of the good life”  \n   Pizzas are served in slices, so the phrase “get a slice of the good life” is both the idiom for enjoying prosperity and a literal description of what every pizza already experiences—being sliced. The humor comes from the pizza wanting the metaphorical version of something it physically can’t escape.\n\nPut together, the punch line pretends the pizza has the same career anxieties as people: needing cash (dough) and dreaming of comfort (a slice of the good life), while

In [29]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it wanted to make some *dough* and finally get a *slice* of the good life!', 'explanation': 'The joke is built on two layers of pizza-flavored wordplay:\n\n1. “Dough”  \n   In pizza terms, dough is literally what the crust is made of. In slang, “dough” has meant “money” since at least the 19th century. So the pizza is applying for a job to earn the thing it’s already made of—an identity gag.\n\n2. “A slice of the good life”  \n   Pizzas are served in slices, so the phrase “get a slice of the good life” is both the idiom for enjoying prosperity and a literal description of what every pizza already experiences—being sliced. The humor comes from the pizza wanting the metaphorical version of something it physically can’t escape.\n\nPut together, the punch line pretends the pizza has the same career anxieties as people: needing cash (dough) and dreaming of comfort (a slice of the good life), whil

In [30]:
len(list(workflow.get_state_history(config1))) # no. of checkpoints

4

In [31]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the pasta get promoted?\n\nBecause it always *used its noodle* and *stuck to the job*—no matter how *boiling* things got!',
 'explanation': 'The joke is a triple-decker pun built around pasta clichés:\n\n1. “Used its noodle” – “Noodle” is already slang for “head,” so the pasta is literally using its own body and figuratively using its brain.  \n2. “Stuck to the job” – Pasta is starchy, so it literally sticks to pots (and itself) while cooking; metaphorically it means persistence.  \n3. “Boiling” – Water boils when pasta cooks; workplace situations also get “boiling” (intense).\n\nPromote the pasta: it keeps a cool head while literally sitting in boiling water and never lets go—exactly the kind of employee bosses say they want.'}

In [32]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it wanted to make some *dough* and finally get a *slice* of the good life!', 'explanation': 'The joke is built on two layers of pizza-flavored wordplay:\n\n1. “Dough”  \n   In pizza terms, dough is literally what the crust is made of. In slang, “dough” has meant “money” since at least the 19th century. So the pizza is applying for a job to earn the thing it’s already made of—an identity gag.\n\n2. “A slice of the good life”  \n   Pizzas are served in slices, so the phrase “get a slice of the good life” is both the idiom for enjoying prosperity and a literal description of what every pizza already experiences—being sliced. The humor comes from the pizza wanting the metaphorical version of something it physically can’t escape.\n\nPut together, the punch line pretends the pizza has the same career anxieties as people: needing cash (dough) and dreaming of comfort (a slice of the good life), while

In [34]:
listofworkflow = list(workflow.get_state_history(config1))

for workflows in listofworkflow:
    print(workflows)
    print('-'*50)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it wanted to make some *dough* and finally get a *slice* of the good life!', 'explanation': 'The joke is built on two layers of pizza-flavored wordplay:\n\n1. “Dough”  \n   In pizza terms, dough is literally what the crust is made of. In slang, “dough” has meant “money” since at least the 19th century. So the pizza is applying for a job to earn the thing it’s already made of—an identity gag.\n\n2. “A slice of the good life”  \n   Pizzas are served in slices, so the phrase “get a slice of the good life” is both the idiom for enjoying prosperity and a literal description of what every pizza already experiences—being sliced. The humor comes from the pizza wanting the metaphorical version of something it physically can’t escape.\n\nPut together, the punch line pretends the pizza has the same career anxieties as people: needing cash (dough) and dreaming of comfort (a slice of the good life), while

## benfits of persistence

### Time Travel

In [35]:
workflow.get_state({'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1336c6-cbf1-6cb4-8000-bce5d28fe7a3'}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1336c6-cbf1-6cb4-8000-bce5d28fe7a3'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-04-08T17:00:19.914462+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1336c6-cbec-6d12-bfff-09a2687622d2'}}, tasks=(PregelTask(id='73781f6c-49f7-d0a7-7f39-9e2e1fc0c4b4', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the pizza apply for a job?\n\nBecause it wanted to make some *dough* and finally get a *slice* of the good life!'}),), interrupts=())

In [36]:
workflow.invoke(None, {'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1336c6-cbf1-6cb4-8000-bce5d28fe7a3'}})

{'topic': 'pizza',
 'joke': 'Why did the pizza apply for a job?\n\nBecause it wanted to make some *dough*!',
 'explanation': 'The joke is a pun on the word “dough.”  \nLiterally, pizza is made from dough, so it already “has” dough.  \nIn slang, “dough” also means money.  \nBy “applying for a job,” the pizza is trying to earn money—i.e., to “make dough.”  \nSo the humor comes from the same word describing both the pizza’s body and the paycheck it hopes to take home.'}

In [37]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it wanted to make some *dough*!', 'explanation': 'The joke is a pun on the word “dough.”  \nLiterally, pizza is made from dough, so it already “has” dough.  \nIn slang, “dough” also means money.  \nBy “applying for a job,” the pizza is trying to earn money—i.e., to “make dough.”  \nSo the humor comes from the same word describing both the pizza’s body and the paycheck it hopes to take home.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1336cb-0d51-6f6e-8002-50258fddc9d8'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-04-08T17:02:14.143780+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1336cb-049b-6f2f-8001-d3b1f0094df7'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it wanted to make som

#### Updating State

In [ ]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f06cc72-ca16-6359-8001-7eea05e07dd2'}}

In [ ]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc72-ca16-6359-8001-7eea05e07dd2'}}, metadata={'source': 'update', 'step': 1, 'parents': {}, 'thread_id': '1'}, created_at='2025-07-29T21:58:35.155132+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, tasks=(PregelTask(id='0f085bb0-c1e8-d9fd-fb15-c427126b7cd6', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the mushroom go to the pizza party? Because he was a fungi and everyone wanted a pizza him!', 'explanation': 'This joke plays on the word "fun guy" (fungi) which sounds like "fungi," a type of mushroom. The play on words is that the mushroom went to the pizza party because he was a "fun guy" and people 

In [ ]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc72-ca16-6359-8001-7eea05e07dd2"}})

{'topic': 'samosa',
 'joke': 'Why did the samosa bring a ladder to the party? \nBecause it wanted to be the best snack in the room and rise to the occasion!',
 'explanation': 'This joke plays on the double meaning of the word "rise." In one sense, "rise" means to physically move upwards, which is why the samosa brought a ladder to the party. However, in another sense, "rise" can also mean to perform well or excel, as in rising to the occasion. So, the samosa brought a ladder to symbolize its desire to physically rise above the other snacks at the party and also to metaphorically rise to the occasion by being the best snack in the room.'}

In [ ]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa bring a ladder to the party? \nBecause it wanted to be the best snack in the room and rise to the occasion!', 'explanation': 'This joke plays on the double meaning of the word "rise." In one sense, "rise" means to physically move upwards, which is why the samosa brought a ladder to the party. However, in another sense, "rise" can also mean to perform well or excel, as in rising to the occasion. So, the samosa brought a ladder to symbolize its desire to physically rise above the other snacks at the party and also to metaphorically rise to the occasion by being the best snack in the room.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc75-4407-6195-8003-b08dcfd27511'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}, 'thread_id': '1'}, created_at='2025-07-29T21:59:41.628661+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoin

### Fault Tolerance

In [14]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [15]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [16]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [17]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)


In [1]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)


🔁 Re-running the graph to demonstrate fault tolerance...


NameError: name 'graph' is not defined

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))